In [0]:
# Definindo schema
catalog_name = "cinedata_analytics"
schema_silver = "silver"
schema_bronze = "bronze"

# Criando schema se não existir
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_silver}")

# Lendo dados
df_origem = spark.read.table(f"{catalog_name}.{schema_bronze}.tb_movies_info")

# Renomeando colunas
df_silver = (
    df_origem
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
)

In [0]:
from pyspark.sql.functions import col, trim, upper, regexp_replace, initcap, when

# Normalizando: maiúsculo, removendo hífens sobressalentes, espaços, e formato título
df_silver = df_silver.withColumn(
    "status_filme", initcap(trim(regexp_replace(upper(col("status_filme")), "-+", " ")))
)

# Normalizando: traduzindo status do filme para português e em caso de status desconhecido, informando não informado
df_silver = df_silver.withColumn(
    "status_filme",
    when(col("status_filme") == "Released", "Lançado")
    .when(col("status_filme") == "Post Production", "Pós-Produção")
    .when(col("status_filme") == "In Production", "Em Produção")
    .when(col("status_filme") == "Planned", "Planejado")
    .when(col("status_filme") == "Rumored", "Rumores")
    .when(col("status_filme") == "Canceled", "Cancelado")
    .otherwise("Não Informado")
)

In [0]:
from pyspark.sql.functions import try_to_date, coalesce, year

# Convertendo campo de data lancamento testanto os diferentes padroes e caso não consiga converter, informando como campo null
df_silver = df_silver.withColumn(
    "data_lancamento",
    coalesce(
        try_to_date(col("data_lancamento"), "yyyy-MM-dd"),
        try_to_date(col("data_lancamento"), "MM-dd-yyyy"),
        try_to_date(col("data_lancamento"), "dd/MM/yyyy")
    )
)

# Extraindo somente o ano e criando uma coluna com o ano
df_silver = df_silver.withColumn(
    "ano_lancamento", year(col("data_lancamento"))
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

# Definindo 
janela = Window.partitionBy("id_filme").orderBy(col("ingestion_datetime").desc())

# Mantendo somente a última versão da tabela
df_silver = (
    df_silver.withColumn(
        "linha_numero", row_number().over(janela))
        .filter(col("linha_numero") == 1)
        .drop("linha_numero")
    )
    

In [0]:
(
    df_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_silver}.tb_info_filmes")
)

print("Tabela tb_info_filmes recriada com sucesso.")

In [0]:
# Checagem de ids unicos para identificar duplicidades
df_check = spark.table(f"{catalog_name}.{schema_silver}.tb_info_filmes")
print("Total:", df_check.count())
print("Únicos:", df_check.select("id_filme").distinct().count())